<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Feature Detection and Object Tracking — Requirements</b></h1>
</div>

This notebook validates the exact runtime prerequisites used by `main.ipynb`. It checks the Python runtime, package pins, imports detected from the implementation notebook, required input data, and output-path readiness.


## Python version


In [ ]:
import sys
print(sys.version)
print("Executable:", sys.executable)


## Required packages


In [ ]:
import ast
import json
import sys
from importlib import import_module, metadata
from pathlib import Path

MODULE_REL = Path("labs/computer-vision/feature-tracking")

def locate_lab_root():
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        for candidate in (base, base / MODULE_REL):
            if (
                (candidate / "notebooks" / "main.ipynb").is_file()
                and (candidate / "requirements.txt").is_file()
                and (candidate / "data").is_dir()
            ):
                return candidate
    raise FileNotFoundError(
        f"Could not locate lab root for {MODULE_REL} from {cwd}"
    )

LAB_ROOT = locate_lab_root()
MAIN_NOTEBOOK = LAB_ROOT / "notebooks" / "main.ipynb"
REQUIREMENTS_TXT = LAB_ROOT / "requirements.txt"

MODULE_TO_DISTRIBUTION = {
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "cv2": "opencv-python",
    "torch": "torch",
    "mnist": "python-mnist",
    "tqdm": "tqdm",
    "PIL": "Pillow",
    "scipy": "scipy",
}
DISTRIBUTION_TO_MODULE = {
    distribution: module
    for module, distribution in MODULE_TO_DISTRIBUTION.items()
}
DISTRIBUTION_TO_MODULE["ipykernel"] = "ipykernel"

main_doc = json.loads(MAIN_NOTEBOOK.read_text(encoding="utf-8"))
imports = set()
for cell in main_doc.get("cells", []):
    if cell.get("cell_type") != "code":
        continue
    try:
        tree = ast.parse("".join(cell.get("source", [])))
    except SyntaxError:
        continue
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                imports.add(alias.name.split(".")[0])
        elif isinstance(node, ast.ImportFrom) and node.module:
            imports.add(node.module.split(".")[0])

stdlib = set(getattr(sys, "stdlib_module_names", ()))
third_party = sorted(
    module for module in imports
    if module not in stdlib and module != "__future__"
)
unmapped_modules = [
    module for module in third_party
    if module not in MODULE_TO_DISTRIBUTION
]

pins = {}
unpinned_lines = []
for raw_line in REQUIREMENTS_TXT.read_text(encoding="utf-8").splitlines():
    line = raw_line.strip()
    if not line or line.startswith("#"):
        continue
    if "==" not in line:
        unpinned_lines.append(line)
        continue
    distribution, version = line.split("==", 1)
    pins[distribution.strip()] = version.strip()

expected_distributions = {
    MODULE_TO_DISTRIBUTION[module]
    for module in third_party
    if module in MODULE_TO_DISTRIBUTION
}
expected_distributions.add("ipykernel")

missing_specs = sorted(expected_distributions - set(pins))
extra_specs = sorted(set(pins) - expected_distributions)

package_rows = []
for distribution in sorted(expected_distributions):
    module_name = DISTRIBUTION_TO_MODULE[distribution]
    expected_version = pins.get(distribution)
    try:
        import_module(module_name)
        installed_version = metadata.version(distribution)
        import_ok = True
    except Exception as exc:
        installed_version = f"ERROR: {exc}"
        import_ok = False
    version_ok = (
        import_ok
        and expected_version is not None
        and installed_version == expected_version
    )
    package_rows.append(
        (distribution, module_name, expected_version, installed_version, version_ok)
    )

print("Third-party imports detected in main.ipynb:")
for module in third_party:
    print(f"  - {module}")

print("\nPackage verification:")
for distribution, module_name, expected, installed, ok in package_rows:
    print(
        f"  {distribution:<18} import={module_name:<12} "
        f"expected={str(expected):<12} installed={installed} "
        f"{'PASS' if ok else 'FAIL'}"
    )

if unmapped_modules:
    print("Unmapped third-party imports:", unmapped_modules)
if missing_specs:
    print("Missing requirements.txt entries:", missing_specs)
if extra_specs:
    print("Extra requirements.txt entries:", extra_specs)
if unpinned_lines:
    print("Unpinned requirement lines:", unpinned_lines)

PACKAGES_OK = (
    not unmapped_modules
    and not missing_specs
    and not extra_specs
    and not unpinned_lines
    and all(row[-1] for row in package_rows)
)
print("\nPackage alignment:", "PASS" if PACKAGES_OK else "FAIL")


## Required data files


In [ ]:
DATA_CONFIG = {"mode":"video","files":["data/video1.mp4"]}
DATA_DIR = LAB_ROOT / "data"

def nonempty_file(path):
    return path.is_file() and path.stat().st_size > 0

DATA_OK = True

if DATA_CONFIG["mode"] == "video":
    rel_path = DATA_CONFIG["files"][0]
    path = LAB_ROOT / rel_path
    ok = nonempty_file(path)
    if ok:
        try:
            cv2 = import_module("cv2")
            cap = cv2.VideoCapture(str(path))
            opened = cap.isOpened()
            read_ok, frame = cap.read()
            cap.release()
            ok = opened and read_ok and frame is not None and frame.size > 0
        except Exception:
            ok = False
    DATA_OK &= ok
    print(f"{rel_path}: {'PASS' if ok else 'FAIL'}")

elif DATA_CONFIG["mode"] == "images":
    try:
        Image = import_module("PIL.Image")
    except Exception as exc:
        Image = None
        DATA_OK = False
        print("Pillow validation unavailable:", exc)

    for rel_path in DATA_CONFIG["files"]:
        path = LAB_ROOT / rel_path
        ok = nonempty_file(path)
        if ok and Image is not None:
            try:
                with Image.open(path) as image:
                    image.verify()
            except Exception:
                ok = False
        DATA_OK &= ok
        print(f"{rel_path}: {'PASS' if ok else 'FAIL'}")
else:
    raise ValueError(f"Unsupported DATA_CONFIG mode: {DATA_CONFIG['mode']}")

print("\nData readiness:", "PASS" if DATA_OK else "FAIL")


## Installation

If any package check fails, install the lab environment from the lab root:

```bash
python -m pip install -r requirements.txt
```

Do not run `main.ipynb` until the package and data checks above pass.


## Environment summary


In [ ]:
import platform

OUTPUT_DIR = LAB_ROOT / "outputs" / "figures"
MAIN_OK = MAIN_NOTEBOOK.is_file()

try:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    probe = OUTPUT_DIR / ".write_probe"
    probe.write_text("ok", encoding="utf-8")
    probe.unlink()
    OUTPUT_OK = True
except Exception as exc:
    OUTPUT_OK = False
    print("Output-directory check failed:", exc)

main_source = "\n".join(
    "".join(cell.get("source", []))
    for cell in main_doc.get("cells", [])
    if cell.get("cell_type") == "code"
)
cwd_sensitive = (
    'Path("../data' in main_source
    or 'Path("../outputs' in main_source
)
cwd = Path.cwd().resolve()
notebooks_cwd = (LAB_ROOT / "notebooks").resolve()
lab_cwd = LAB_ROOT.resolve()

if cwd_sensitive:
    CWD_OK = cwd == notebooks_cwd
    expected_cwd = notebooks_cwd
else:
    CWD_OK = cwd in {lab_cwd, notebooks_cwd}
    expected_cwd = f"{lab_cwd} or {notebooks_cwd}"

print(f"OS: {platform.system()} {platform.release()}")
print(f"Machine: {platform.machine()}")
print(f"Python executable: {sys.executable}")
print(f"Lab root: {LAB_ROOT}")
print(f"main.ipynb: {'PASS' if MAIN_OK else 'FAIL'}")
print(f"Output directory: {'PASS' if OUTPUT_OK else 'FAIL'}")
print(f"Working-directory alignment: {'PASS' if CWD_OK else 'FAIL'}")
if not CWD_OK:
    print(f"Expected working directory: {expected_cwd}")
    print(f"Current working directory : {cwd}")

READY = PACKAGES_OK and DATA_OK and MAIN_OK and OUTPUT_OK and CWD_OK
print("\nREADY TO RUN main.ipynb:", "PASS" if READY else "FAIL")

if not READY:
    raise RuntimeError(
        "Environment is not fully aligned with main.ipynb. "
        "Resolve the failed checks above before running the implementation."
    )
